# Deep agent: subgraphs as tools

Worktree-local notebook. The compiled graphs stay as they are; this notebook
wires them as LangChain tools and optionally asks a deep agent to choose
among them.

Run Jupyter from this worktree root after `uv sync`.

```text
user request
     ↓
create_deep_agent
     ↓
cohort / classify_* / discover_* / recommend_pathway / persist_recc
     ↓
compiled subgraphs + TaskStore
```

In [7]:
hf auth whoami

SyntaxError: invalid syntax (3173910406.py, line 1)

In [6]:
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-MiniLM-L6-v2")

OSError: sentence-transformers/all-MiniLM-L6-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from dotenv import load_dotenv
from pprint import pprint

print(load_dotenv(".env", override=True))
key = os.getenv("OPENAI_API_KEY")
print(key[:10] if key else "OPENAI_API_KEY not found")

from playbook import (
    SYSTEM_PROMPT,
    classify_intent,
    classify_subflow,
    cohort,
    configure_runtime,
    create_playbook_agent,
    discover_intent,
    discover_subflow,
    persist_recc,
    recommend_pathway,
)
from playbook import runtime as rt
from playbook.fill import EDA_DATA_DIR

ROOT = Path(".").resolve()
# close an existing store connection so this block can be rerun
store = getattr(rt, "store", None)
conn = getattr(getattr(store, "_local", None), "conn", None) if store else None
if conn is not None:
    conn.close()
    store._local.conn = None

playbook, store = configure_runtime(
    data_dir=EDA_DATA_DIR,
    store_path=EDA_DATA_DIR / "run_store.sqlite",
)



print("tasks", len(store.fetchall("SELECT task_id FROM tasks")))
print("intents", playbook.intent_ids())
print("LANGSMITH_TRACING", os.environ.get("LANGSMITH_TRACING"))

True
sk-svcacct
tasks 145
intents ['account_access', 'order_issue']
LANGSMITH_TRACING true


## Direct tool calls

These invoke the compiled graphs with no model. Use this cell to confirm
the SQLite store survives LangGraph worker threads.

In [2]:
cohort_out = cohort.invoke(
    {
        "start_date": "2026-09-01",
        "end_date": "2026-09-07",
        "method": "bert",
        "cohort_query": {},
        "run_id": "notebook-deep-agent",
    }
)
intent_out = classify_intent.invoke({})
subflow_out = classify_subflow.invoke({})

print(f"cohort output\n")
pprint(cohort_out)
print("\nIntent output")
pprint(intent_out)
print("\nsubflow output")
pprint(subflow_out)

cohort output

{'cohort_summary': {'end': '2026-09-07',
                    'max_conversation_date': '2026-09-07',
                    'method': 'bert',
                    'min_conversation_date': '2026-09-01',
                    'n': 51,
                    'start': '2026-09-01',
                    'task_ids': ['1035',
                                 '1054',
                                 '1067',
                                 '1074',
                                 '1092',
                                 '1131',
                                 '1148',
                                 '1152',
                                 '1156',
                                 '1223',
                                 '1232',
                                 '1265',
                                 '1303',
                                 '1333',
                                 '1336',
                                 '1361',
                                 '1593',
                   

Discovery and pathway tools. `recommend_pathway` drafts only;
`persist_recc` writes the store / KB.

In [3]:
print("------------\ndiscover_intent")
pprint(discover_intent.invoke({}))
print("\n------------\ndiscover_subflow")
pprint(discover_subflow.invoke({}))
print("\n------------\nrecommend_pathway")
pprint(recommend_pathway.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\persist_recc_pathway")
pprint(persist_recc.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\nlist reccs")
pprint(store.list_recommendations(rt.current_run_id))
print("\n------------\nkb version")
print("kb_version", playbook.version)

------------
discover_intent
{'approved_change_ids': ['prop_0be18f17'],
 'current_stage': 'summarize_intent_discovery',
 'discovery_summary': {'intents': {'approved_count': 1,
                                   'candidate_count': 1,
                                   'changes': [{'candidate': 'shipping_issue',
                                                'decision': 'accept',
                                                'kb_version': 2,
                                                'n_tasks': 34,
                                                'parent': None,
                                                'proposal_id': 'prop_0be18f17',
                                                'type': 'new_intent'},
                                               {'candidate': None,
                                                'decision': 'decline',
                                                'kb_version': None,
                                                'n_tasks': 2,
       

### current seeded data
seed: ~Aug 25–Sep 8

noise: Sep 9–10

status_payment_method: Sep 10–12

slow_speed: Sep 12–14

## Deep agent (optional)

Needs `OPENAI_API_KEY`. The agent picks tools from the request instead of
following a fixed classify → discover → recommend graph.

In [4]:
thread_id="notebook-deep-agent"
def user_message(agent, content, thread_id):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        content
                    ),
                }
            ]
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    #print(result["messages"][-1].content)
    result["messages"][-1].pretty_print()

In [5]:
from langchain.chat_models import init_chat_model

if os.environ.get("OPENAI_API_KEY"):
    model = init_chat_model("openai:gpt-4.1-mini", temperature=0)
    agent = create_playbook_agent(model=model)
    # or construct it yourself:
    # agent = create_deep_agent(model=model, tools=[...], system_prompt=SYSTEM_PROMPT)
    content = """
    "Classify the 2026-09-01 to 2026-09-07 cohort with jaccard. 
    Stop after classification unless unresolved tasks clearly need discovery.
    """
    user_message(agent, content, thread_id)    
else:
    print("Set OPENAI_API_KEY to invoke create_playbook_agent.")
    print("System prompt starts with:")
    print(SYSTEM_PROMPT.splitlines()[0])

OSError: sentence-transformers/all-MiniLM-L6-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [16]:
content = """yes, run discovery over this cohort"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Discovery on the unresolved tasks in the cohort identified a new candidate intent named "shipping_issue," which has been approved and covers 34 tasks. Two tasks were identified as outliers and the outlier proposal was declined.

Would you like me to proceed with classifying these tasks under the new "shipping_issue" intent or take any other action?


In [11]:
content = """what is the name of the new intent?"""
user_message(agent, content, thread_id)

I could not find the name of the new intent from the discovery proposal ID in the knowledge base files. It appears the new intent is approved but the name is not directly accessible from the current files.

I can provide a summary of the discovered intent from the discovery process or attempt to retrieve the candidate intent name from the discovery metadata if you want. How would you like to proceed?
================================== Ai Message ==================================
Name: playbook_deep_agent

I could not find the name of the new intent from the discovery proposal ID in the knowledge base files. It appears the new intent is approved but the name is not directly accessible from the current files.

I can provide a summary of the discovered intent from the discovery process or attempt to retrieve the candidate intent name from the discovery metadata if you want. How would you like to proceed?


In [17]:
content = """what are the available intent labels in the kb?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The available intent labels in the knowledge base are:

1. Account Access: Covers username, password, and two-factor authentication issues. Subflows include Recover Username, Recover Password, and Reset Two-Factor Auth.

2. Order Issue: Covers getting the status of an order or changing an order, possibly related to shipping. Subflows include Status Mystery Fee, Status Delivery Time, Manage Upgrade, and Manage Cancel.

3. Shipping Issue: Covers package shipment problems such as delivered missing items, carrier issues, porch delivery, tracking, order, purchase, and replacement. This is a newly approved intent with no subflows yet.

Let me know if you want to take any further actions with these intents.


In [18]:
content = """yes, classify unresolved tasks with the new intent"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The unresolved tasks in the cohort have been classified with the new "shipping_issue" intent for 34 tasks. There are still 2 tasks that remain unknown and unresolved.

Would you like me to investigate these remaining unknown tasks further or take any other action?


In [19]:
content = """classify empty subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

All tasks in the cohort currently lack subflow assignments. None of the tasks have been classified into existing subflows yet, so 49 tasks remain unknown and unresolved at the subflow level.

Would you like me to proceed with discovering potential new subflows within the intents or take any other action?


In [20]:
content = """what are the existing subflows in the kb?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The existing subflows in the knowledge base are:

For the "Account Access" intent:
- Recover Username
- Recover Password
- Reset Two-Factor Auth

For the "Order Issue" intent:
- Status Mystery Fee
- Status Delivery Time
- Manage Upgrade
- Manage Cancel

For the "Shipping Issue" intent:
- There are currently no subflows defined.

Would you like me to discover potential new subflows for the "Shipping Issue" intent or any other intent?


In [21]:
content = """go ahead and classify using existing subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The classification attempt using existing subflows did not assign any subflows to the tasks in the cohort. All 49 tasks remain without subflow classification.

Since the existing subflows do not cover these tasks, would you like me to proceed with discovering new subflows within the intents?


In [22]:

content = """why were none classified to existing subflows?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The reason none of the tasks were classified to existing subflows is that the tasks in the "shipping_issue" intent do not match any existing subflows, as there are currently no subflows defined for the "shipping_issue" intent in the knowledge base.

For the "order_issue" intent, the existing subflows have specific action patterns such as "pull-up-account," "verify-identity," "ask-the-oracle," "shipping-status," "update-order," and others. However, the sample tasks from the cohort do not align well enough with these subflow action patterns to be classified confidently.

In summary:
- "Shipping Issue" intent has no subflows defined yet.
- "Order Issue" intent subflows have specific action sequences that the tasks do not sufficiently match.

Would you like me to proceed with discovering new subflows for these intents, especially for "shipping_issue"?


In [23]:
content = """yes, discover new subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Discovery for new subflows within the "shipping_issue" and "order_issue" intents found two emerging subflow candidates, one for each intent. However, both emerging subflow proposals were declined and no new subflows were approved.

Therefore, no new subflows will be added to the knowledge base at this time.

Is there anything else you would like to do?


In [24]:
content = """tell me about the emerging subflow proposals"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The emerging subflow proposals "prop_efba1912" and "prop_870f3c6b" do not have detailed guidance or descriptions available in the knowledge base. They were identified as emerging subflows for the "shipping_issue" and "order_issue" intents respectively, but both proposals were declined and no new subflows were approved.

The current knowledge base shows no subflows under "shipping_issue" and four subflows under "order_issue" (Status Mystery Fee, Status Delivery Time, Manage Upgrade, Manage Cancel), but these emerging proposals did not result in additions.

If you want, I can help analyze the tasks further or assist with other actions.


In [25]:
content = """can you get task summaries for a sample from prop_efba1912"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Here are summaries for a sample of tasks from the emerging subflow proposal "prop_efba1912" under the "shipping_issue" intent:

1. Task 1035 (2026-09-06):
- Customer wants to upgrade shipping to overnight for their last order.
- Agent attempts to verify account and order IDs but cannot pull them up.
- Customer confirms shipping status is "in transit."
- Agent informs it's too late to upgrade shipping but order likely arrives soon.
- Intent: shipping_issue, Subflow: unknown.

2. Task 1054 (2026-09-06):
- Customer wants to upgrade order to overnight shipping.
- Agent verifies account and order ID.
- Shipping status is "in transit," so upgrade is not possible.
- Agent reassures order will arrive soon.
- Intent: shipping_issue, Subflow: unknown.

These tasks involve customers requesting shipping upgrades for orders already in transit, with agents explaining the limitations and provid